# Yoruba OCR - Kaggle GPU runner

Use this notebook on Kaggle with a GPU accelerator. Attach two Kaggle datasets when possible:

- repo dataset: a copy of this repository, default slug `yoruba-ocr-research`
- data dataset: processed OCR data, default slug `yoruba-ocr-data`

If you do not attach the repo as a dataset, set `GITHUB_REPO` and enable Internet in Kaggle notebook settings.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
PROJECT_ROOT = KAGGLE_WORKING / 'yoruba_ocr_research'

REPO_INPUT_SLUG = os.environ.get('KAGGLE_REPO_INPUT', 'yoruba-ocr-research')
DATA_INPUT_SLUG = os.environ.get('KAGGLE_DATA_INPUT', 'yoruba-ocr-data')
GITHUB_REPO = os.environ.get('GITHUB_REPO', '')

if not PROJECT_ROOT.exists():
    repo_input = KAGGLE_INPUT / REPO_INPUT_SLUG
    if repo_input.exists():
        print('Copying repo from Kaggle input:', repo_input)
        shutil.copytree(
            repo_input,
            PROJECT_ROOT,
            ignore=shutil.ignore_patterns('.git', '.hf_cache', '__pycache__', 'results/tables/archive'),
        )
    elif Path.cwd().name == 'yoruba_ocr_research':
        PROJECT_ROOT = Path.cwd()
    elif GITHUB_REPO:
        print('Cloning repo:', GITHUB_REPO)
        subprocess.check_call(['git', 'clone', GITHUB_REPO, str(PROJECT_ROOT)])
    else:
        raise RuntimeError('Attach repo dataset or set GITHUB_REPO with Internet enabled.')

os.chdir(PROJECT_ROOT)
os.environ['PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['PYTHON'] = sys.executable
os.environ['HF_HOME'] = str(KAGGLE_WORKING / '.hf_cache')
os.environ['HF_HUB_CACHE'] = os.environ['HF_HOME']
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'disabled'
print('PROJECT_ROOT =', PROJECT_ROOT)
print('Python =', sys.executable)

## Link processed data from `/kaggle/input`

The pipeline expects `data/processed`. This cell finds either `<dataset>/data/processed` or `<dataset>/processed` and symlinks it into the working repo.

In [ ]:
processed = PROJECT_ROOT / 'data' / 'processed'
if not processed.exists():
    candidates = [
        KAGGLE_INPUT / DATA_INPUT_SLUG / 'data' / 'processed',
        KAGGLE_INPUT / DATA_INPUT_SLUG / 'processed',
    ]
    candidates += sorted(KAGGLE_INPUT.glob('*/data/processed'))
    candidates += sorted(KAGGLE_INPUT.glob('*/processed'))
    source = next((p for p in candidates if p.exists()), None)
    if source is None:
        raise RuntimeError('Could not find processed data under /kaggle/input. Attach the data dataset.')
    (PROJECT_ROOT / 'data').mkdir(exist_ok=True)
    processed.symlink_to(source, target_is_directory=True)
    print('Linked data/processed ->', source)
else:
    print('Found existing data/processed:', processed)

for rel in ['labels/train.txt', 'labels/val.txt', 'labels/test.txt', 'dictionary/yoruba_char_dict.txt']:
    path = processed / rel
    if not path.exists():
        raise FileNotFoundError(path)
print('Data layout OK')

## Install GPU/runtime dependencies

Kaggle must have Internet enabled for first-time installs and Hugging Face downloads. If Paddle GPU install fails, set `PADDLE_PIP_SPEC` to the exact wheel spec for the current CUDA image.

In [ ]:
def pip_install(*packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

pip_install('-U', 'pip')
pip_install(os.environ.get('PADDLE_PIP_SPEC', 'paddlepaddle-gpu>=2.6,<2.7'))
pip_install('-r', 'requirements.txt')

paddle_dir = PROJECT_ROOT / 'PaddleOCR'
if not paddle_dir.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/PaddlePaddle/PaddleOCR.git', 'PaddleOCR'])
pip_install('-r', 'PaddleOCR/requirements.txt')

# Reassert the VLM stack last; PaddleOCR dependencies can downgrade transitive packages.
pip_install('-U', 'transformers>=5.0.0', 'accelerate>=1.1.0', 'huggingface_hub>=1.5.0', 'datasets', 'safetensors', 'einops', 'torchvision', 'bitsandbytes')

import paddle, torch, transformers, accelerate
print('Paddle CUDA devices:', paddle.device.cuda.device_count())
print('Torch CUDA:', torch.cuda.is_available())
print('Transformers:', transformers.__version__)
if int(transformers.__version__.split('.')[0]) < 5:
    raise RuntimeError('Restart the Kaggle session, then rerun setup. transformers>=5 is required.')

## Choose what to run

Start with zero-shot baselines. Turn on SFT only when the GPU session has enough time and disk.

In [ ]:
RUN_RESET = True
RUN_PADDLE_BASELINE = True
RUN_VLM_ZERO_SHOT = True
RUN_PADDLEOCRVL16_SFT = False
RUN_ANALYSIS = True

os.environ['EVAL_USE_GPU'] = '1'
os.environ['CONFIG_FORCE_GPU'] = '1'
os.environ['PADDLEOCRVL16_QUANTIZE_4BIT'] = os.environ.get('PADDLEOCRVL16_QUANTIZE_4BIT', '0')
os.environ['GLM_QUANTIZE_4BIT'] = os.environ.get('GLM_QUANTIZE_4BIT', '0')

def run(cmd):
    print('$', ' '.join(cmd))
    subprocess.check_call(cmd, cwd=PROJECT_ROOT, env=os.environ.copy())

In [ ]:
if RUN_RESET:
    run([sys.executable, 'scripts/metrics_lifecycle.py', 'reset'])

run(['bash', 'scripts/shell/phase_02_analyze.sh'])
run(['bash', 'scripts/shell/phase_03_config.sh'])

if RUN_PADDLE_BASELINE:
    run(['bash', 'scripts/shell/phase_05_eval_paddleocr_recognition.sh'])

if RUN_VLM_ZERO_SHOT:
    run(['bash', 'scripts/shell/phase_15_eval_paddleocrvl16_zero_shot.sh'])
    run(['bash', 'scripts/shell/phase_18_eval_glm_ocr_zero_shot.sh'])

if RUN_PADDLEOCRVL16_SFT:
    run(['bash', 'scripts/shell/phase_14_export_paddleocrvl16_sft.sh'])
    run(['bash', 'scripts/shell/phase_16_train_paddleocrvl16_sft.sh'])
    run(['bash', 'scripts/shell/phase_17_eval_paddleocrvl16_sft.sh'])

if RUN_ANALYSIS:
    for script in ['17_stratified_error_analysis.py', '18_der_universe_ablation.py', '19_bootstrap_metric_cis.py']:
        run([sys.executable, f'scripts/{script}'])
    run([sys.executable, 'scripts/11_compile_results.py'])
    run([sys.executable, 'scripts/22_generate_plots.py'])
    run([sys.executable, 'scripts/23_write_research_approach.py', '--output', 'research_approach.md'])

## Export Kaggle outputs

Kaggle persists files under `/kaggle/working` as notebook output artifacts.

In [ ]:
export_root = KAGGLE_WORKING / 'yoruba_ocr_outputs'
if export_root.exists():
    shutil.rmtree(export_root)
export_root.mkdir(parents=True)
for name in ['results', 'research_approach.md']:
    src = PROJECT_ROOT / name
    if src.is_dir():
        shutil.copytree(src, export_root / name)
    elif src.is_file():
        shutil.copy2(src, export_root / name)
if (PROJECT_ROOT / 'experiments').exists() and RUN_PADDLEOCRVL16_SFT:
    shutil.copytree(PROJECT_ROOT / 'experiments', export_root / 'experiments')
archive = shutil.make_archive(str(export_root), 'zip', export_root)
print('Exported:', archive)